
# <u>V3-SEA-8 Dataset Workshop - Time Series `tas` (Air Surface Temperature) Plot</u>

This notebook demonstrates how to use V3-SEA-8 `tas` data to compare historical and future annual mean air temperature change over Malaysia.

## Introduction to the climate data used:
The V3 dataset contains climate projection data based on an ensemble of CMIP6 global climate models (GCMs) `ACCESS-CM2`, `EC-Earth3`, `MIROC6`, `MPI-ESM1-2-HR`, `NorESM2-MM`, `UKESM1-0-LL`. In this folder, the data are organised by model, then by scenario, and finally by climate variable files.

## Designed Workflow:
The workflow is intentionally written step by step for guiding:

1. Import used libraries and workshop configurations
2. Helper functions defining as preparation
3. Inspect the climate-data folder structure
4. Open an example NetCDF file
5. Load regional Malaysia shapefiles
6. Mask model grids to a selected region
7. Compute annual area-mean TAS
8. Compute the 1995-2014 baseline
9. Calculate anomalies relative to the baseline
10. Plot one model or a multi-model ensemble

## 0. Local IDE Setup

This notebook is designed to run from a local Jupyter environment, such as VS Code, JupyterLab, or classic Jupyter Notebook. Run the terminal setup from `README.md` first, then run the cells in order.

Google Colab is not recommended for the raw NetCDF workflow because mounted Google Drive can fail while streaming large HDF5/NetCDF files.


Run this cell first to confirm that the notebook is running in the expected local folder and that the required Python packages can be imported.


In [ ]:
import sys
from pathlib import Path

WORKSHOP_PATH = Path.cwd().resolve()
if str(WORKSHOP_PATH) not in sys.path:
    sys.path.insert(0, str(WORKSHOP_PATH))

required_packages = {
    "xarray": "xarray",
    "netCDF4": "netCDF4",
    "geopandas": "geopandas",
    "shapely": "shapely",
    "dask": "dask",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}

missing_packages = []
for package_name, import_name in required_packages.items():
    try:
        __import__(import_name)
    except ImportError:
        missing_packages.append(package_name)

if missing_packages:
    raise ModuleNotFoundError(
        "Missing required package(s): "
        + ", ".join(missing_packages)
        + ". Install them with `pip install -r requirements.txt` or `conda env create -f environment.yml`."
    )

print("Local workshop setup looks ready.")
print(f"  Working directory: {WORKSHOP_PATH}")
print(f"  Python executable: {sys.executable}")
print("  Required packages can be imported.")


## 1. Import Libraries And Configuration Settings

- Import necessary libraries through this workshop 
- Manipulation by changing `REGION_KEY` and `MODELS_TO_RUN` to run different evaluation (control runtime):
    * Use one model for a fast first run.
    * Use two or more models to show ensemble spread.
    * Use `"all"` to run every available model.

In [ ]:
from __future__ import annotations

import importlib.util
import re
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.ticker import MultipleLocator
import numpy as np
import pandas as pd
import xarray as xr

# Load validated data paths from the local workshop config.
# Edit config.py or set V3SEA8_CCRS / V3SEA8_SHAPES if your folders are elsewhere.
try:
    from config import CCRS_DATA_PATH, SHAPEFILES_PATH
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "config.py was not found. Start Jupyter from the repository folder that contains this notebook and config.py."
    ) from exc

WORKSHOP_DIR = Path.cwd()
CCRS_ROOT = CCRS_DATA_PATH
SHAPES_ROOT = SHAPEFILES_PATH
OUTPUT_DIR = WORKSHOP_DIR / "outputs" / "tas"

REGION_KEY = "sabah"  # choose: "penisular", "sabah", or "sarawak"
MODELS_TO_RUN = ["ACCESS-CM2", "MIROC6"]  # use one model, multiple models, or "all"
CHUNKS_TIME = 12
USE_CACHE = True

SCENARIOS = ("historical", "ssp126", "ssp245", "ssp585")
FUTURE_SCENARIOS = ("ssp126", "ssp245", "ssp585")
SCENARIO_LABELS = {
    "historical": "Historical",
    "ssp126": "SSP1-2.6",
    "ssp245": "SSP2-4.5",
    "ssp585": "SSP5-8.5",
}
SCENARIO_COLORS = {
    "historical": "#3c3c3c",
    "ssp126": "#2b6cb0",
    "ssp245": "#dd8a1a",
    "ssp585": "#c0392b",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Workshop configuration loaded successfully")
print(f"  Region: {REGION_KEY}")
print(f"  Models: {MODELS_TO_RUN}")
print(f"  CCRS data: {CCRS_ROOT}")
print(f"  Shapefiles: {SHAPES_ROOT}")
print(f"  Output directory: {OUTPUT_DIR}")

if not CCRS_ROOT.exists():
    raise FileNotFoundError(f"CCRS data folder not found: {CCRS_ROOT}")
if not SHAPES_ROOT.exists():
    raise FileNotFoundError(f"Shapefile folder not found: {SHAPES_ROOT}")


## 2. Define Regions And Helper Functions

- Each Malaysia region is represented by one or more shapefiles. 
- The notebook merges the given shapefile geometries into one regional boundary, then uses that boundary to mask the model grid.

In [ ]:
@dataclass(frozen=True)
class Region:
    key: str
    title: str
    shapefiles: tuple[str, ...]


REGIONS = {
    "penisular": Region(
        key="penisular",
        title="Peninsular Malaysia",
        shapefiles=(
            "PENINSULAR_MALAYSIA/PM_Shapefile_Zone1/PM_Northwest_Zone1.shp",
            "PENINSULAR_MALAYSIA/PM_Shapefile_Zone2/PM_East_Coast_Zone2.shp",
            "PENINSULAR_MALAYSIA/PM_Shapefile_Zone3/PM_Southwest_Zone3.shp",
        ),
    ),
    "sabah": Region(
        key="sabah",
        title="Sabah",
        shapefiles=(
            "SABAH/SBH_Shapefile_Zone1/Sabah_Northwest_Zone1.shp",
            "SABAH/SBH_Shapefile_Zone2/Sabah_Southeast_Zone2.shp",
        ),
    ),
    "sarawak": Region(
        key="sarawak",
        title="Sarawak",
        shapefiles=(
            "SARAWAK/SRWK_Shapefile_Zone1/Sarawak_West.shp",
            "SARAWAK/SRWK_Shapefile_Zone2/Sarawak_Inland2.shp",
        ),
    ),
}

region = REGIONS[REGION_KEY]
region

In [ ]:
# Files fetching from specific models and scenarios:
def scenario_files(model_dir: Path, scenario: str) -> list[Path]:
    return sorted((model_dir / scenario).glob("**/tas/*.nc"))


def all_model_dirs(ccrs_root: Path) -> list[Path]:
    return sorted(path for path in ccrs_root.iterdir() if path.is_dir())


def selected_model_dirs(ccrs_root: Path, models_to_run: str | list[str]) -> list[Path]:
    available = {path.name: path for path in all_model_dirs(ccrs_root)}
    if models_to_run == "all":
        return [available[name] for name in sorted(available)]
    missing = [name for name in models_to_run if name not in available]
    if missing:
        raise FileNotFoundError(f"Selected model folder(s) not found: {missing}")
    return [available[name] for name in models_to_run]


# Time Series Extracting (Year Label)
def year_from_filename(path: Path) -> int:
    match = re.search(r"_(\d{4})\d{2}-\d{4}\d{2}\.nc$", path.name)
    if not match:
        raise ValueError(f"Cannot parse year from {path}")
    return int(match.group(1))

# Sanity check for the coordinate slicing (when lat & lon not in order)
def coord_slice(coord: xr.DataArray, lower: float, upper: float) -> slice:
    if float(coord[0]) > float(coord[-1]):
        return slice(upper, lower)
    return slice(lower, upper)

# coord_slice is optional, if lat and lon guaranteed to be in order, da.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

### Caching Shapefiles: Reducing Redundant Computation

In this workshop, we load and merge regional shapefiles many times:
- To display the regional boundary (Section 4)
- To mask climate grids (Section 5)
- During annual area-mean calculations (Section 6)

Without caching, each call would re-read files from disk and re-compute the geometry merge—expensive operations for large datasets. 

**The `@lru_cache` decorator solves this:** It automatically stores the merged geometry in memory after the first call. Subsequent calls with the same region argument return the cached result instantly, avoiding redundant disk I/O and geometry merging. This simple technique dramatically reduces runtime and memory pressure, especially when processing multiple models and scenarios.

In [ ]:
# Load and merge all shapefiles for a region into one unified geometry.
# This function reads multiple zone shapefiles, ensures they use WGS84 coordinates,
# and combines them using unary_union. The result is cached to avoid re-reading
# files and re-computing the geometry merge on repeated calls.

@lru_cache(maxsize=None)
def load_region_geometry(region_key: str):
    """Load and merge shapefiles for a selected region, returning a unified geometry."""
    import geopandas as gpd
    from shapely.ops import unary_union

    selected_region = REGIONS[region_key]
    geometries = []
    
    for relative_path in selected_region.shapefiles:
        shape_path = SHAPES_ROOT / relative_path
        if not shape_path.exists():
            raise FileNotFoundError(f"Missing shapefile: {shape_path}")
        
        gdf = gpd.read_file(shape_path)
        
        # Ensure WGS84 coordinates
        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:4326")
        elif gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs("EPSG:4326")
        
        geometries.extend(geom for geom in gdf.geometry if geom is not None and not geom.is_empty)

    # Merge all zone geometries into one unified boundary
    geometry = unary_union(geometries)
    if not geometry.is_valid:
        geometry = geometry.buffer(0)
    
    return geometry

## 3. Inspect the Data Structure

Before doing any full computation, we first inspect the available models and open one sample `tas` file. This allows us to check how the data are stored, what scenarios and variables are available, and whether the file dimensions, coordinates, and metadata are suitable for analysis.

In [ ]:
models = selected_model_dirs(CCRS_ROOT, MODELS_TO_RUN)
print("Models selected:")
for model in models:
    print(f"- {model.name}")

print("\nFile counts by model/scenario:")
for model in models:
    counts = {scenario: len(scenario_files(model, scenario)) for scenario in SCENARIOS}
    print(model.name, counts)

In [ ]:
sample_file = scenario_files(models[0], "historical")[0]
print(sample_file)

with xr.open_dataset(sample_file, engine="netcdf4") as sample_ds:
    display(sample_ds)

## 4. Load And Preview The Shapefile Boundary

The shapefiles are loaded with GeoPandas, converted to WGS84 longitude/latitude if needed, then merged into one geometry for the selected region.

In [ ]:
geometry = load_region_geometry(REGION_KEY)
print(region.title)
bounds = geometry.bounds
print(f"Bounds: {bounds}")
print(f"  West (min lon):  {bounds[0]:.6f}°")
print(f"  South (min lat): {bounds[1]:.6f}°")
print(f"  East (max lon):  {bounds[2]:.6f}°")
print(f"  North (max lat): {bounds[3]:.6f}°")

In [ ]:
import geopandas as gpd

boundary_gdf = gpd.GeoDataFrame(geometry=[geometry], crs="EPSG:4326")
ax = boundary_gdf.boundary.plot(figsize=(5, 4), color="black")
ax.set_title(f"{region.title} shapefile boundary")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

## 5. Mask The Climate Grid To The Region

The mask keeps only grid cells whose longitude/latitude points fall inside the selected shapefile geometry. Before masking, the data is cropped to the shapefile bounding box to reduce memory and computation.

In [ ]:
def mask_region(da: xr.DataArray, region_key: str) -> xr.DataArray:
    from shapely import contains_xy

    geometry = load_region_geometry(region_key)
    lon_min, lat_min, lon_max, lat_max = geometry.bounds

    lat_step = float(np.abs(np.diff(da.lat.values)).min()) if da.sizes["lat"] > 1 else 0.0
    lon_step = float(np.abs(np.diff(da.lon.values)).min()) if da.sizes["lon"] > 1 else 0.0

    cropped = da.sel(
        lat=coord_slice(da.lat, lat_min - lat_step, lat_max + lat_step),
        lon=coord_slice(da.lon, lon_min - lon_step, lon_max + lon_step),
    )

    lon2d, lat2d = np.meshgrid(cropped.lon.values, cropped.lat.values)
    inside = contains_xy(geometry.buffer(1e-12), lon2d, lat2d)
    if int(inside.sum()) == 0:
        raise ValueError("The shapefile mask did not select any grid cells.")

    mask = xr.DataArray(
        inside,
        coords={"lat": cropped.lat, "lon": cropped.lon},
        dims=("lat", "lon"),
        name=f"{region_key}_mask",
    )
    return cropped.where(mask)

In [ ]:
# using the historical data to investigate the shape file union masking
with xr.open_dataset(sample_file, engine="netcdf4") as sample_ds:
    masked_sample = mask_region(sample_ds["tas"].isel(time=0), REGION_KEY)

masked_sample.plot(figsize=(6, 4))
plt.title(f"Example masked TAS grid: {region.title}")
plt.show()

## 6. Compute Annual Area-Mean TAS

This function opens monthly TAS files lazily with Dask, which breaks large datasets into manageable chunks and processes them on-demand—avoiding the need to load entire files into memory at once. This dramatically reduces runtime and memory footprint when handling many large NetCDF files.

The function converts Kelvin to Celsius, masks the selected region, calculates annual means, then calculates a spatial mean across all selected grid cells.

If `time_bnds` exists, monthly values are weighted by the length of each month.

In [ ]:
def annual_area_mean(files: list[Path], region_key: str, chunks_time: int = 12) -> pd.Series:
    ds = xr.open_mfdataset(
        [str(path) for path in files],
        combine="by_coords",
        chunks={"time": chunks_time}, 
        data_vars="minimal",
        coords="minimal",
        compat="override",
        parallel=True,
        engine="netcdf4",
    )
    try:
        tas = mask_region(ds["tas"], region_key)

        units = str(tas.attrs.get("units", "")).lower()
        if units in {"k", "kelvin"}:
            tas = tas - 273.15

        if "time_bnds" in ds and ds["time_bnds"].sizes.get("bnds") == 2:
            month_weights = ds["time_bnds"].isel(bnds=1) - ds["time_bnds"].isel(bnds=0)
            month_weights = month_weights.astype("timedelta64[s]").astype(float)
            numerator = (tas * month_weights).groupby("time.year").sum("time", skipna=False)
            denominator = month_weights.groupby("time.year").sum("time")
            annual = numerator / denominator
        else:
            annual = tas.groupby("time.year").mean("time", skipna=True)

        series = annual.mean(("lat", "lon"), skipna=True).compute().to_series()
        series.index = series.index.astype(int)
        return series.sort_index()
    finally:
        ds.close()

## 7. Cache Or Compute Model Time Series

The first run can be slow. The notebook saves one CSV per region/model/scenario, so repeated workshop runs can reuse the processed annual regional mean values.

In [ ]:
CACHE_VERSION = "workshop_shape_mask_v1"


def read_or_compute_series(model_dir: Path, scenario: str) -> pd.Series:
    cache_dir = OUTPUT_DIR / "cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{REGION_KEY}_{CACHE_VERSION}_{model_dir.name}_{scenario}_annual_tas_c.csv"

    if USE_CACHE and cache_path.exists():
        cached = pd.read_csv(cache_path)
        return pd.Series(cached["tas_c"].to_numpy(), index=cached["year"].astype(int))

    files = scenario_files(model_dir, scenario)
    if not files:
        raise FileNotFoundError(f"No TAS files found for {model_dir.name} {scenario}")

    series = annual_area_mean(files, REGION_KEY, CHUNKS_TIME)
    series.rename("tas_c").reset_index().rename(columns={"year": "year"}).to_csv(cache_path, index=False)
    return series


def build_anomaly_table(model_dirs_to_run: list[Path]) -> pd.DataFrame:
    rows = []
    for model_dir in model_dirs_to_run:
        print(f"Processing baseline for {model_dir.name}")
        historical = read_or_compute_series(model_dir, "historical")
        baseline = historical.loc[1995:2014].mean()
        if np.isnan(baseline):
            raise ValueError(f"Missing 1995-2014 baseline for {model_dir.name}")

        for scenario in SCENARIOS:
            print(f"  {model_dir.name} {scenario}")
            series = read_or_compute_series(model_dir, scenario)
            for year, tas_c in series.items():
                rows.append(
                    {
                        "region": REGION_KEY,
                        "model": model_dir.name,
                        "scenario": scenario,
                        "year": int(year),
                        "tas_c": float(tas_c),
                        "baseline_1995_2014_c": float(baseline),
                        "anomaly_c": float(tas_c - baseline),
                    }
                )

    df = pd.DataFrame(rows).sort_values(["scenario", "model", "year"])
    out_path = OUTPUT_DIR / f"tas_{REGION_KEY}_workshop_model_anomalies.csv"
    df.to_csv(out_path, index=False)
    print(f"Wrote {out_path}")
    return df

## 8. Run The TAS Calculation

For each selected model, the 1995-2014 historical mean is used as that model's baseline. Future values are then shown as change relative to that model-specific baseline.

In [ ]:
df = build_anomaly_table(models)
df.head()

## 9. Ensemble Summaries

- When multiple models are selected, the notebook summarizes the model ensemble with a mean, minimum, and maximum for each year. 
- When one model is selected, the minimum, mean, and maximum are the same, so the plot will show a single model line without shading.

In [ ]:
def ensemble_summary(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby(["scenario", "year"], as_index=False)["anomaly_c"]
        .agg(mean="mean", minimum="min", maximum="max", model_count="count")
        .sort_values(["scenario", "year"])
    )


def scenario_plot_summary(summary: pd.DataFrame, scenario: str) -> pd.DataFrame:
    scenario_summary = summary[summary["scenario"] == scenario].copy()
    if scenario == "historical":
        return scenario_summary

    historical_anchor = summary[(summary["scenario"] == "historical") & (summary["year"] == 2014)].copy()
    if historical_anchor.empty:
        return scenario_summary
    historical_anchor.loc[:, "scenario"] = scenario
    return pd.concat([historical_anchor, scenario_summary], ignore_index=True).sort_values("year")


def late_century_min_mean_max(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for scenario in FUTURE_SCENARIOS:
        per_model = (
            df[(df["scenario"] == scenario) & (df["year"] >= 2080) & (df["year"] <= 2099)]
            .groupby("model")["anomaly_c"]
            .mean()
            .dropna()
        )
        rows.append(
            {
                "scenario": scenario,
                "minimum": per_model.min(),
                "mean": per_model.mean(),
                "maximum": per_model.max(),
                "model_count": int(per_model.count()),
            }
        )
    return pd.DataFrame(rows)


summary = ensemble_summary(df)
summary.to_csv(OUTPUT_DIR / f"tas_{REGION_KEY}_workshop_ensemble_summary.csv", index=False)
summary.head()

## 10. Plot The Result

The plot adapts to the number of selected models:

- one model: the line is that model's anomaly; no shaded spread is drawn
- multiple models: the line is the ensemble mean; the shaded band is the minimum-to-maximum model spread

In [ ]:
def plot_tas_workshop(df: pd.DataFrame, region: Region) -> Path:
    summary = ensemble_summary(df)
    model_count = df["model"].nunique()
    draw_spread = model_count > 1

    fig = plt.figure(figsize=(13, 6.2), constrained_layout=True)
    grid = fig.add_gridspec(nrows=1, ncols=2, width_ratios=[4.3, 1.25])
    ax = fig.add_subplot(grid[0, 0])
    range_ax = fig.add_subplot(grid[0, 1], sharey=ax)

    for scenario in SCENARIOS:
        scenario_summary = scenario_plot_summary(summary, scenario)
        if scenario_summary.empty:
            continue
        years = scenario_summary["year"].to_numpy()
        color = SCENARIO_COLORS[scenario]
        if draw_spread:
            ax.fill_between(
                years,
                scenario_summary["minimum"].to_numpy(),
                scenario_summary["maximum"].to_numpy(),
                color=color,
                alpha=0.16,
                linewidth=0,
            )
        ax.plot(
            years,
            scenario_summary["mean"].to_numpy(),
            color=color,
            linewidth=2.1,
            label=SCENARIO_LABELS[scenario],
        )

    ax.axhline(0, color="#555555", linestyle=":", linewidth=1.4)
    ax.set_xlabel("Year", fontsize=14)
    ax.set_ylabel("Mean Air Temperature Change (deg C)", fontsize=14)
    ax.set_xlim(1955, 2099)
    ax.xaxis.set_major_locator(MultipleLocator(20))
    ax.xaxis.set_minor_locator(MultipleLocator(5))
    ax.tick_params(axis="x", which="minor", length=3)
    ax.grid(True, axis="y", color="#d9d9d9", linewidth=0.7, alpha=0.8)
    ax.legend(frameon=False, ncols=4, loc="upper left")

    late = late_century_min_mean_max(df)
    x_positions = np.arange(len(late)) + 1
    for x_pos, row in zip(x_positions, late.itertuples(index=False)):
        color = SCENARIO_COLORS[row.scenario]
        width = 0.45
        if draw_spread:
            range_ax.add_patch(
                Rectangle(
                    (x_pos - width / 2, row.minimum),
                    width,
                    row.maximum - row.minimum,
                    facecolor=color,
                    edgecolor="none",
                    alpha=0.55,
                    zorder=2,
                )
            )
        range_ax.hlines(
            y=row.mean,
            xmin=x_pos - width / 2,
            xmax=x_pos + width / 2,
            color=color,
            linewidth=2.4,
            zorder=3,
        )

    range_ax.axhline(0, color="#555555", linestyle=":", linewidth=1.4)
    range_ax.set_title("2080-2099\nmodel spread" if draw_spread else "2080-2099\nsingle model")
    range_ax.set_xticks(x_positions)
    range_ax.set_xticklabels([SCENARIO_LABELS[s] for s in FUTURE_SCENARIOS], rotation=20, ha="right")
    range_ax.grid(True, axis="y", color="#d9d9d9", linewidth=0.7, alpha=0.8)
    range_ax.tick_params(axis="y", labelleft=True)
    range_ax.set_xlim(0.4, len(late) + 0.6)

    ymin, ymax = ax.get_ylim()
    range_ax.set_ylim(ymin, ymax)

    model_text = "single model" if model_count == 1 else f"{model_count} models"
    fig.suptitle(
        f"{region.title}: Annual Mean Air Temperature Change relative to 1995-2014 ({model_text})",
        y=1.03,
        fontsize=16,
    )
    output_path = OUTPUT_DIR / f"tas_{REGION_KEY}_workshop_annual_mean_change.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    return output_path


plot_path = plot_tas_workshop(df, region)
plot_path

## 11. Discussion Prompts

- How do the three SSP scenarios differ after mid-century?
- Does model spread increase over time?
- How does the selected region affect the magnitude of projected warming?
- What changes when `MODELS_TO_RUN` is switched from one model to multiple models?
- Why is the baseline period important when interpreting anomalies?